# Generate Audio File from example sentences

In [ ]:
import os
import wave

from dotenv import load_dotenv
from google import genai
from google.genai import types

load_dotenv()
GOOGLE_API_KEY = os.getenv("GOOGLE_API_KEY")
client = genai.Client(api_key=GOOGLE_API_KEY)


def wave_file(filename, pcm, channels=1, rate=24000, sample_width=2):
    """Set up the wave file to save the output"""
    with wave.open(filename, "wb") as wf:
        wf.setnchannels(channels)
        wf.setsampwidth(sample_width)
        wf.setframerate(rate)
        wf.writeframes(pcm)


def generate_audio_for_word(content: str):
    """Generate audio for a given word using Gemini TTS"""
    response = client.models.generate_content(
        model="gemini-2.5-flash-preview-tts",
        contents=content,
        config=types.GenerateContentConfig(
            response_modalities=["AUDIO"],
            speech_config=types.SpeechConfig(
                voice_config=types.VoiceConfig(
                    prebuilt_voice_config=types.PrebuiltVoiceConfig(
                        voice_name="Kore",
                    )
                )
            ),
        ),
    )
    data = response.candidates[0].content.parts[0].inline_data.data
    return data

In [ ]:
from langchain.prompts import PromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.rate_limiters import InMemoryRateLimiter
from langchain_google_genai import GoogleGenerativeAI

rate_limiter = InMemoryRateLimiter(
    requests_per_second=0.1,  # <-- Super slow! We can only make a request once every 10 seconds!!
    check_every_n_seconds=0.1,  # Wake up every 100 ms to check whether allowed to make a request,
    max_bucket_size=10,  # Controls the maximum burst size.
)

# Gemini APIの初期化
llm = GoogleGenerativeAI(
    model="gemini-2.0-flash", google_api_key=GOOGLE_API_KEY, rate_limiter=rate_limiter
)

# 例文生成用のプロンプトテンプレート
example_prompt = PromptTemplate.from_template(
    """Generate the simple one sentence discription of the English word "{word}" in Japanese.
        
    # example for `comprise`:
    (全体が部分を)含む、構成する

    # example for `accrue`:
    (利子、利益、負債などが) 累積する、増える、生じる
    """
)

# Chain
chain = example_prompt | llm | StrOutputParser()


def generate_short_explanation(word: str) -> str:
    """単語に対して例文を生成する"""
    output = chain.invoke({"word": word})
    return output


def generate_content_from_db(word: str, examples: list[str]) -> str:
    """Generate content from the database for a given word and examples"""
    japanese_explanation: str = generate_short_explanation(word)
    content = f"""TTS the following sentence. Please read English word and examples as an native English speaker, while Japanese explanation should be read as a native Japanese speaker.:
    # sentence
    {word} {japanese_explanation}
    {examples[0]}
    {examples[1]}
    {examples[2]}
    """
    return content

In [ ]:
generate_short_explanation("patter")

'(小雨などが)ぱらぱらと降る、または早口でぺらぺらと話す。'

In [ ]:
import time

import pandas as pd

df_word = pd.read_csv("../word_data/lv7.csv")
df_example = pd.read_csv("../generate_examples/lv7_with_examples.csv")

for index, row in df_word.iterrows():
    # APIリクエスト制限を考慮して少し待機
    time.sleep(10)
    examples = df_example[df_example["word_id"] == row["word_id"]]["example"].tolist()
    content = generate_content_from_db(row["word"], examples)
    print(row["word_id"], content)
    audio_data = generate_audio_for_word(content)
    file_name = f"{row['word_id']}.wav"
    wave_file(file_name, audio_data)
    print("======")

6001 TTS the following sentence. Please read English word and examples as an native English speaker, while Japanese explanation should be read as a native Japanese speaker.:
    # sentence
    symposium (特定のテーマについて) 専門家が集まり意見交換や発表を行う会議、討論会
    The university is hosting a symposium on climate change next month, featuring experts from around the world.
    She presented her research at the international symposium on artificial intelligence.
    Attendance at the symposium is mandatory for all graduate students in the biology department.
    


ClientError: 429 RESOURCE_EXHAUSTED. {'error': {'code': 429, 'message': 'You exceeded your current quota, please check your plan and billing details. For more information on this error, head to: https://ai.google.dev/gemini-api/docs/rate-limits.', 'status': 'RESOURCE_EXHAUSTED', 'details': [{'@type': 'type.googleapis.com/google.rpc.QuotaFailure', 'violations': [{'quotaMetric': 'generativelanguage.googleapis.com/generate_content_free_tier_requests', 'quotaId': 'GenerateRequestsPerDayPerProjectPerModel-FreeTier', 'quotaDimensions': {'location': 'global', 'model': 'gemini-2.5-flash-tts'}, 'quotaValue': '15'}]}, {'@type': 'type.googleapis.com/google.rpc.Help', 'links': [{'description': 'Learn more about Gemini API quotas', 'url': 'https://ai.google.dev/gemini-api/docs/rate-limits'}]}, {'@type': 'type.googleapis.com/google.rpc.RetryInfo', 'retryDelay': '38s'}]}}

In [ ]:
file_name = "out.wav"
wave_file(file_name, data)  # Saves the file to current directory